In [ ]:
!nvidia-smi
!pip -q install -U ultralytics roboflow

Tue Mar 10 23:19:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from roboflow import Roboflow
import os

ROBOFLOW_API_KEY = "GETYOUROWN"

WORKSPACE = "atathamuscoinsdataset"
PROJECT = "u.s.-coins-dataset-a.tatham"

# Choose a dataset version.
# Universe shows many versions; v20 is listed as the most recent. :contentReference[oaicite:3]{index=3}
VERSION = 20

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print("Dataset downloaded to:", dataset.location)
print("Contents:", os.listdir(dataset.location))

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to U.S.-Coins-Dataset---A.Tatham-20 in yolov8:: 100%|██████████| 4796/4796 [00:00<00:00, 6878.50it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset downloaded to: /content/U.S.-Coins-Dataset---A.Tatham-20
Contents: ['valid', 'data.yaml', 'README.roboflow.txt', 'train', 'README.dataset.txt', 'test']


In [ ]:
import yaml, os

data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

data_yaml

{'names': ['Dime', 'Nickel', 'Penny', 'Quarter'],
 'nc': 4,
 'roboflow': {'license': 'CC BY 4.0',
  'project': 'u.s.-coins-dataset-a.tatham',
  'url': 'https://universe.roboflow.com/atathamuscoinsdataset/u.s.-coins-dataset-a.tatham/dataset/20',
  'version': 20,
  'workspace': 'atathamuscoinsdataset'},
 'test': '../test/images',
 'train': '../train/images',
 'val': '../valid/images'}

In [ ]:
from ultralytics import YOLO

# A good starting point: yolov8n.pt (fast, small).
# You can try yolov8s.pt for better accuracy (slower).
model = YOLO("yolov8n.pt")

results = model.train(
    data=data_yaml_path,
    imgsz=640,
    epochs=50,
    batch=16,
    device=0,
    project="coin_counter",
    name="yolov8n_us_coins",
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/U.S.-Coins-Dataset---A.Tatham-20/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_us_coins, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

In [ ]:
val_metrics = model.val(data=data_yaml_path, imgsz=640, device=0)
val_metrics

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1398.1±486.3 MB/s, size: 48.3 KB)
val: Scanning /content/U.S.-Coins-Dataset---A.Tatham-20/valid/labels.cache... 260 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 260/260 83.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 17/17 2.0it/s 8.6s
                   all        260       8876      0.952      0.889      0.924      0.735
                  Dime        172       2509      0.972      0.841      0.901      0.689
                Nickel        165       1974      0.907      0.843      0.883       0.71
                 Penny        208       2901      0.979      0.957      0.973      0.763
               Quarter        123       1492      0.949      0.914      0.938       0.78
Speed: 3.6ms preprocess,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d9a36f19e50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
glob.glob("**/weights/best.pt", recursive=True)[:20]

Mounted at /content/drive


['runs/detect/coin_counter/yolov8n_us_coins/weights/best.pt']

In [ ]:
import os, shutil

best_path = "runs/detect/coin_counter/yolov8n_us_coins/weights/best.pt"
drive_dir = "/content/drive/MyDrive/CoinCounter"
os.makedirs(drive_dir, exist_ok=True)

shutil.copy(best_path, os.path.join(drive_dir, "best.pt"))
print("Saved to:", os.path.join(drive_dir, "best.pt"))

Saved to: /content/drive/MyDrive/CoinCounter/best.pt
